# 01 · Feature selection con ABC
**Colab:** Entorno de ejecución → Ejecutar todo.

- Partícula = máscara de ceros y unos (uso / no uso la columna)
- Aptitud = accuracy CV − multa por usar muchas columnas
- Rival = SelectKBest (filtro, no es al azar)


In [ ]:
# Motor ABC + PSO (incluido para que Colab no dependa de rutas)
#!/usr/bin/env python3
"""Primitivas ABC y PSO usadas por los cuatro ejercicios."""

import numpy as np


def pso_minimize(objective, bounds, n_particles=20, iters=25, w=0.7, c1=1.5, c2=1.5, seed=42):
    """PSO global-best. Minimiza objective(x).

    Ciclo: representacion = posicion continua,
    inicializacion uniforme, aptitud = objective,
    comportamiento = inercia + pbest + gbest,
    evolucion por iteraciones, parada = iters.
    """
    rng = np.random.default_rng(seed)
    lo, hi = np.asarray(bounds[0], float), np.asarray(bounds[1], float)
    dim = lo.size
    pos = rng.uniform(lo, hi, size=(n_particles, dim))
    vel = np.zeros_like(pos)
    costs = np.array([objective(p) for p in pos])
    pbest, pbest_c = pos.copy(), costs.copy()
    g = int(np.argmin(pbest_c))
    gbest, gbest_c = pbest[g].copy(), float(pbest_c[g])
    hist = [gbest_c]
    swarm_hist = [pos.copy()]
    for _ in range(iters):
        r1, r2 = rng.random(pos.shape), rng.random(pos.shape)
        vel = w * vel + c1 * r1 * (pbest - pos) + c2 * r2 * (gbest - pos)
        pos = np.clip(pos + vel, lo, hi)
        costs = np.array([objective(p) for p in pos])
        improved = costs < pbest_c
        pbest[improved] = pos[improved]
        pbest_c[improved] = costs[improved]
        g = int(np.argmin(pbest_c))
        if pbest_c[g] < gbest_c:
            gbest, gbest_c = pbest[g].copy(), float(pbest_c[g])
        hist.append(gbest_c)
        swarm_hist.append(pos.copy())
    return gbest, gbest_c, np.array(hist), swarm_hist


def abc_binary_maximize(objective, n_bits, n_bees=10, cycles=12, limit=4, seed=42):
    """ABC binario. Maximiza objective(bitstring).

    Ciclo: representacion = fuente de alimento binaria,
    inicializacion aleatoria, aptitud = objective,
    comportamiento = empleada / observadora / exploradora,
    evolucion por ciclos, parada = cycles.
    """
    rng = np.random.default_rng(seed)
    foods = rng.integers(0, 2, size=(n_bees, n_bits))
    empty = foods.sum(axis=1) == 0
    if empty.any():
        foods[empty, rng.integers(0, n_bits, size=int(empty.sum()))] = 1
    fit = np.array([objective(f) for f in foods])
    trials = np.zeros(n_bees, dtype=int)
    best_i = int(np.argmax(fit))
    best, best_f = foods[best_i].copy(), float(fit[best_i])
    hist = [best_f]

    def neighbor(src):
        k = int(rng.integers(0, n_bits))
        nxt = src.copy()
        nxt[k] ^= 1
        if nxt.sum() == 0:
            nxt[k] = 1
        return nxt

    for _ in range(cycles):
        for i in range(n_bees):
            cand = neighbor(foods[i])
            fc = objective(cand)
            if fc >= fit[i]:
                foods[i], fit[i], trials[i] = cand, fc, 0
            else:
                trials[i] += 1
        probs = fit - fit.min() + 1e-9
        probs = probs / probs.sum()
        for _o in range(n_bees):
            i = int(rng.choice(n_bees, p=probs))
            cand = neighbor(foods[i])
            fc = objective(cand)
            if fc >= fit[i]:
                foods[i], fit[i], trials[i] = cand, fc, 0
            else:
                trials[i] += 1
        for i in range(n_bees):
            if trials[i] >= limit:
                foods[i] = rng.integers(0, 2, size=n_bits)
                if foods[i].sum() == 0:
                    foods[i][int(rng.integers(0, n_bits))] = 1
                fit[i] = objective(foods[i])
                trials[i] = 0
        bi = int(np.argmax(fit))
        if fit[bi] > best_f:
            best, best_f = foods[bi].copy(), float(fit[bi])
        hist.append(best_f)
    return best, best_f, np.array(hist)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})


In [ ]:
def pack_xy(bunch):
    X = StandardScaler().fit_transform(bunch.data)
    y = bunch.target
    names = np.array(bunch.feature_names)
    return X, y, names

def fitness_factory(X, y, clf, cv, lam):
    def eval_mask(mask):
        mask = np.asarray(mask, dtype=bool)
        if mask.sum() == 0:
            return 0.0
        acc = cross_val_score(clf, X[:, mask], y, cv=cv, scoring="accuracy").mean()
        return float(acc - lam * (mask.sum() / mask.size))
    return eval_mask


## Breast Cancer — 30 columnas


In [ ]:
X, y, names = pack_xy(load_breast_cancer())
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
clf = LogisticRegression(max_iter=200, solver="liblinear")
fit_fn = fitness_factory(X, y, clf, cv, lam=0.012)

acc_all = cross_val_score(clf, X, y, cv=cv, scoring="accuracy").mean()
mask, best_fit, hist = abc_binary_maximize(fit_fn, n_bits=X.shape[1], n_bees=8, cycles=8, limit=4, seed=SEED)
acc_abc = cross_val_score(clf, X[:, mask.astype(bool)], y, cv=cv, scoring="accuracy").mean()
k = max(int(mask.sum()), 1)
acc_skb = cross_val_score(clf, SelectKBest(f_classif, k=k).fit_transform(X, y), y, cv=cv, scoring="accuracy").mean()

print(f"30 -> {int(mask.sum())} columnas")
print(f"todas={acc_all:.4f}  ABC={acc_abc:.4f}  SelectKBest={acc_skb:.4f}")
print("ABC eligió:", ", ".join(names[mask.astype(bool)]))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist, marker="o", color="#3ee0e8")
ax[0].set_title("Convergencia ABC"); ax[0].set_xlabel("ciclo"); ax[0].set_ylabel("aptitud")
ax[1].barh(np.arange(len(names)), mask, color=np.where(mask, "#3ee0e8", "#333"))
ax[1].set_yticks(np.arange(len(names))); ax[1].set_yticklabels(names, fontsize=6)
ax[1].set_title("Máscara (cian = usada)")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.4))
labs = [f"Todas ({X.shape[1]})", f"ABC ({int(mask.sum())})", f"SelectKBest ({k})"]
vals = [acc_all, acc_abc, acc_skb]
bars = ax.bar(labs, vals, color=["#666", "#3ee0e8", "#4c78a8"])
ax.set_ylim(min(vals)-0.04, 1); ax.set_ylabel("accuracy CV")
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.002, f"{v:.3f}", ha="center")
plt.tight_layout(); plt.show()


## Wine — corroboración (13 columnas)


In [ ]:
Xw, yw, nw = pack_xy(load_wine())
clfw = LogisticRegression(max_iter=250)
fit_w = fitness_factory(Xw, yw, clfw, 3, lam=0.015)
acc_w_all = cross_val_score(clfw, Xw, yw, cv=3, scoring="accuracy").mean()
mw, _, _ = abc_binary_maximize(fit_w, n_bits=Xw.shape[1], n_bees=8, cycles=8, seed=7)
acc_w_abc = cross_val_score(clfw, Xw[:, mw.astype(bool)], yw, cv=3, scoring="accuracy").mean()
print(f"Wine 13 -> {int(mw.sum())}")
print(f"todas={acc_w_all:.4f}  ABC={acc_w_abc:.4f}")
print("ABC eligió:", ", ".join(nw[mw.astype(bool)]))
